In [69]:
import os
import json
import pandas as pd
import traceback

In [70]:
from langchain_openai import ChatOpenAI

In [71]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

True

In [72]:
KEY = os.getenv("OPENAI_API_KEY")

In [73]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5
)

In [74]:
llm

ChatOpenAI(profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000021A8355E250>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021A8355E410>, root_client=<openai.OpenAI object at 0x0000021A8355E010>, root_async_client=<openai.AsyncOpenAI object at 0x0000021A8355E3D0>, model_name='gpt-4o-mini', temperature=0.5, model_kwargs={}, open

In [75]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence
from langchain_community.callbacks import get_openai_callback
import PyPDF2

In [76]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here"
        },
        "correct": "correct answer"
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here"
        },
        "correct": "correct answer"
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here"
        },
        "correct": "correct answer"
    }
}

In [93]:
TEMPLATE = """
Text: {text}

You are an expert MCQ maker. Given the above text, it is your job to create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.

Make sure:
- Questions are not repeated
- Questions are relevant to the text
- Follow the format strictly

Return ONLY valid JSON. Do not include any extra text.

{response_json}
"""

In [94]:
from langchain_core.prompts import PromptTemplate

quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

In [95]:
quiz_chain = quiz_generation_prompt | llm

In [96]:
TEMPLATE2 = """
You are an expert English grammarian and writer.

Given a Multiple Choice Quiz for {subject} students, evaluate the complexity of the questions and provide a complete analysis of the quiz.

- Keep the analysis within 50 words
- If the quiz is not appropriate for the students, update the questions
- Adjust the tone so it perfectly fits the student level

QUIZ_MCQs:
{quiz}

Provide your evaluation and improved version (if needed):
"""

In [97]:
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2
)

In [98]:
review_chain = quiz_evaluation_prompt | llm

In [99]:
def generate_evaluate_chain(inputs):
    quiz_output = quiz_chain.invoke(inputs)
    
    review_output = review_chain.invoke({
        "subject": inputs["subject"],
        "quiz": quiz_output.content
    })
    
    return {
        "quiz": quiz_output.content,
        "review": review_output.content
    }

In [100]:
file_path = r"D:\mcqgen\data.txt"

In [101]:
with open(file_path, 'r', encoding='utf-8') as file:
    TEXT = file.read()

In [102]:
print(TEXT)

The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]

The earliest machine learning program was introduced in the 1950s when Arthur Samuel invented a computer program that calculated the winning chance in checkers for each side, but the history of machine learning roots back to decades of human desire and effort to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] Hebb's model of neurons interacting with one another set a groundwork for how AIs and machine learning algorithms work under nodes, or artificial neurons used by computers to communicate data.[9] Other researchers who have studied human cognitive systems contri

In [103]:
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [104]:
NUMBER = 5
SUBJECT = "machine learning"
TONE = "simple"

In [105]:
with get_openai_callback() as cb:
    response = generate_evaluate_chain({
        "text": TEXT,
        "number": NUMBER,
        "subject": SUBJECT,
        "tone": TONE,
        "response_json": json.dumps(RESPONSE_JSON)
    })

    print(cb)

Tokens Used: 2340
	Prompt Tokens: 1438
		Prompt Tokens Cached: 0
	Completion Tokens: 902
		Reasoning Tokens: 0
Successful Requests: 2
Total Cost (USD): $0.0007568999999999999


In [106]:
response

{'quiz': '{\n  "1": {\n    "mcq": "Who coined the term \'machine learning\'?",\n    "options": {\n      "a": "Donald Hebb",\n      "b": "Nils Nilsson",\n      "c": "Arthur Samuel",\n      "d": "Alan Turing"\n    },\n    "correct": "c"\n  },\n  "2": {\n    "mcq": "What was the name of the early \'learning machine\' developed by Raytheon Company?",\n    "options": {\n      "a": "Cybertron",\n      "b": "Deep Blue",\n      "c": "AlphaGo",\n      "d": "Neural Net"\n    },\n    "correct": "a"\n  },\n  "3": {\n    "mcq": "What type of learning does reinforcement learning focus on?",\n    "options": {\n      "a": "Classification",\n      "b": "Clustering",\n      "c": "Decision making based on previous actions",\n      "d": "Data synthesis"\n    },\n    "correct": "c"\n  },\n  "4": {\n    "mcq": "Which book provided a formal definition of machine learning algorithms?",\n    "options": {\n      "a": "The Organization of Behavior",\n      "b": "Learning Machines",\n      "c": "Computing Machine

In [107]:
quiz=response.get("quiz")

In [108]:
print(quiz)

{
  "1": {
    "mcq": "Who coined the term 'machine learning'?",
    "options": {
      "a": "Donald Hebb",
      "b": "Nils Nilsson",
      "c": "Arthur Samuel",
      "d": "Alan Turing"
    },
    "correct": "c"
  },
  "2": {
    "mcq": "What was the name of the early 'learning machine' developed by Raytheon Company?",
    "options": {
      "a": "Cybertron",
      "b": "Deep Blue",
      "c": "AlphaGo",
      "d": "Neural Net"
    },
    "correct": "a"
  },
  "3": {
    "mcq": "What type of learning does reinforcement learning focus on?",
    "options": {
      "a": "Classification",
      "b": "Clustering",
      "c": "Decision making based on previous actions",
      "d": "Data synthesis"
    },
    "correct": "c"
  },
  "4": {
    "mcq": "Which book provided a formal definition of machine learning algorithms?",
    "options": {
      "a": "The Organization of Behavior",
      "b": "Learning Machines",
      "c": "Computing Machinery and Intelligence",
      "d": "Machine Learning

In [110]:
quiz = json.loads(quiz)

In [112]:
quiz_table_data = []

for key, value in quiz.items():
    mcq = value["mcq"]
    
    options = " | ".join(
        [f"{opt}: {opt_val}" for opt, opt_val in value["options"].items()]
    )
    
    correct = value["correct"]
    
    quiz_table_data.append({
        "MCQ": mcq,
        "Choices": options,
        "Correct": correct
    })

In [116]:
quiz = pd.DataFrame(quiz_table_data)

In [117]:
quiz.to_csv("Machine_learning_quiz.csv", index=False)